# SHAP 

In [1]:
!pip install shap

In [2]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import shap

/home/fateee/Desktop/ulyana/Implementing_a_Local_Explanations_Assessment_Framework/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SEED = 42
NSAMPLES = 5000
K_TOP = 10
BACKGROUND_SIZE = 50

DATA_DIR = Path("data")
MLP_DIR  = Path("artifacts/mlp")
LIME_DIR = Path("artifacts/lime")
OUT_DIR  = Path("artifacts/shap")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_test  = np.load(DATA_DIR / "X_test.npy")
y_test  = np.load(DATA_DIR / "y_test.npy")

meta          = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
feature_names = meta["feature_names"]
class_names   = meta["target_names"]

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, 1),
        )

    def forward(self, x):
        return self.net(x)


class MLPWrapper:
    """Sklearn-compatible wrapper that exposes predict_proba."""

    def __init__(self, mlp_model, device):
        self.model = mlp_model
        self.device = device

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            t = torch.tensor(np.asarray(X, dtype=np.float32)).to(self.device)
            probs1 = torch.sigmoid(self.model(t)).cpu().numpy().flatten()
        return np.column_stack([1.0 - probs1, probs1])


config = json.loads((MLP_DIR / "model_config.json").read_text(encoding="utf-8"))
device = "cuda" if torch.cuda.is_available() else "cpu"
_mlp = MLP(config["input_dim"]).to(device)
_mlp.load_state_dict(torch.load(MLP_DIR / "model_state.pt", map_location=device))
model = MLPWrapper(_mlp, device)

chosen      = json.loads((LIME_DIR / "chosen_test_indices.json").read_text(encoding="utf-8"))
test_indices = [int(i) for i in chosen["chosen_test_indices"]]

len(test_indices), test_indices[:5]

(10, [7, 15, 49, 53, 55])

In [4]:
(OUT_DIR / "chosen_test_indices.json").write_text(
    json.dumps(chosen, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUT_DIR / "chosen_test_indices.json").as_posix()

'artifacts/shap/chosen_test_indices.json'

In [ ]:
# KernelExplainer needs a background dataset that represents the "average" input.
# We sample a small random subset from the training data.
rng    = np.random.default_rng(SEED)
bg_idx = rng.choice(X_train.shape[0], size=min(BACKGROUND_SIZE, X_train.shape[0]), replace=False)
bg_idx = [int(i) for i in bg_idx]
background = X_train[bg_idx]

(OUT_DIR / "background_indices.json").write_text(
    json.dumps({"seed": SEED, "background_size": int(len(bg_idx)), "indices": bg_idx},
               ensure_ascii=False, indent=2),
    encoding="utf-8",
)
background.shape

(50, 30)

In [ ]:
def predict_proba_class1(X):
    """Return P(benign) as a 1-D array — single scalar output for KernelExplainer."""
    return model.predict_proba(X)[:, 1]

explainer = shap.KernelExplainer(predict_proba_class1, background)
explainer.expected_value

0.5981680217607256

In [7]:
(OUT_DIR / "expected_value.json").write_text(
    json.dumps({"expected_value": float(explainer.expected_value)}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUT_DIR / "expected_value.json").as_posix()

'artifacts/shap/expected_value.json'

In [8]:
shap_class1 = []
per_instance = []

for idx in test_indices:
    x = X_test[idx]
    true_y = int(y_test[idx])
    proba = model.predict_proba(x.reshape(1, -1))[0]
    pred_y = int(np.argmax(proba))

    np.random.seed(SEED + int(idx))

    x2d = x.reshape(1, -1)
    sv = explainer.shap_values(x2d, nsamples=NSAMPLES)
    sv1 = np.array(sv).reshape(-1)
    shap_class1.append(sv1)

    top = np.argsort(np.abs(sv1))[-K_TOP:][::-1]
    weights_top = [
        {"feature": feature_names[int(j)], "value": float(x[int(j)]), "shap": float(sv1[int(j)])}
        for j in top
    ]

    per_instance.append(
        {
            "test_index": int(idx),
            "true_y": true_y,
            "pred_y": pred_y,
            "proba": [float(p) for p in proba],
            "topK_class1": weights_top,
        }
    )

    try:
        exp_obj = shap.Explanation(
            values=sv1,
            base_values=float(explainer.expected_value),
            data=x,
            feature_names=feature_names,
        )
        shap.plots.waterfall(exp_obj, max_display=K_TOP, show=False)
        plt.tight_layout()
        png_path = OUT_DIR / f"waterfall_test_{idx:03d}_class1.png"
        plt.savefig(png_path, dpi=150)
        plt.close()
    except Exception as e:
        per_instance[-1]["plot_error"] = str(e)

shap_class1 = np.stack(shap_class1, axis=0)

np.save(OUT_DIR / "shap_values_class1.npy", shap_class1)

summary = {
    "seed": SEED,
    "nsamples": int(NSAMPLES),
    "k_top": int(K_TOP),
    "background_size": int(len(bg_idx)),
    "n_examples": int(len(test_indices)),
    "class_names": class_names,
    "results": per_instance,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

shap_class1.shape

100%|██████████| 1/1 [00:00<00:00,  5.20it/s]


(10, 30)